# ResNet-50 — static pruning (CIFAR-10)

| phase | optimizer | epochs | batch |
|---|---|---|---|
| dense pretrain | SGD 0.01, linear warmup | 100 (patience 20) | 512 |
| I.P. prune | SGD 0.01 | 5 + 10 recovery | 512 |
| BaCP | SGD 0.1, tau 0.15 | 5 + 10 recovery, then AdamW 1e-4 x 50 | 512 |

Protocol from the paper (appendix Table 1, Section 4.1). Static pruning at
0.95 / 0.97 / 0.99 from an **ImageNet-pretrained** model; magnitude / SNIP-it /
WANDA are the I.P. baselines. Dynamic methods (RigL, EAST) are cited from their
papers, not re-run. Sources for every equation: `docs/foundation.md`.

Full paper Table 1 comparators exist for this model; the results cell prints them next to fresh numbers.


In [ ]:
import sys, pathlib

# Find nb_common.py whether the kernel started in this folder or at the repo root.
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')

import nb_common as nb
info = nb.setup()

## Configure

`SMOKE=True` runs every cell below on 2 batches first -- do that once on a new cluster before real training.

In [ ]:
MODEL      = 'resnet50'
SEED       = 1
GPU        = 0
SPARSITIES = (0.95, 0.97, 0.99)
SMOKE      = False
OVERRIDES  = {}       # e.g. dict(epochs=3) to shorten every run below

for phase in ('dense', 'prune', 'bacp'):
    print(f'{phase:>6}: ', {**nb.FAMILIES[MODEL]['base'], **nb.FAMILIES[MODEL][phase]})

## Weights + preflight

Halts before any GPU time is spent if the model is not actually pretrained (`load_weights` fails soft, so a missing checkpoint would otherwise silently train from random init).

In [ ]:
nb.fetch_imagenet_weights(MODEL)
nb.preflight(MODEL, num_classes=nb.FAMILIES[MODEL]['base']['num_classes'])

## Dense baseline (required first)

Every sparse run below starts from this checkpoint (same seed). Re-running skips it if its record exists; delete the record under `results/runs/` to re-arm.

In [ ]:
dense = nb.make_cell(MODEL, 'dense', seed=SEED, smoke=SMOKE, **OVERRIDES)
out = nb.run(dense, gpu=GPU)

## I.P. — magnitude (Han et al. 2015)

Iterative pruning + recovery, the sparse baseline. One run per sparsity level, streamed back to back.

In [ ]:
cells = [nb.make_cell(MODEL, 'prune', seed=SEED, pruner='magnitude', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## I.P. — SNIP-it (Lee et al. 2019 / Verdenius et al. 2020)

Iterative pruning + recovery, the sparse baseline. One run per sparsity level, streamed back to back.

In [ ]:
cells = [nb.make_cell(MODEL, 'prune', seed=SEED, pruner='snip', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## I.P. — WANDA (Sun et al. 2023)

Iterative pruning + recovery, the sparse baseline. One run per sparsity level, streamed back to back.

In [ ]:
cells = [nb.make_cell(MODEL, 'prune', seed=SEED, pruner='wanda', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## BaCP — magnitude

The contrastive objective (PrC/SnC/FiC + CE, lambdas 0.25 each, tau 0.15 -- `docs/foundation.md` SS2-3), then AdamW finetune.

In [ ]:
cells = [nb.make_cell(MODEL, 'bacp', seed=SEED, pruner='magnitude', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## BaCP — SNIP-it

In [ ]:
cells = [nb.make_cell(MODEL, 'bacp', seed=SEED, pruner='snip', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## BaCP — WANDA

In [ ]:
cells = [nb.make_cell(MODEL, 'bacp', seed=SEED, pruner='wanda', sparsity=s,
                     smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(cells, gpu=GPU)

## Results vs the paper

`-` = not run yet (ours) or not published (paper).

In [ ]:
nb.results_table(MODEL)

## Health

Every static record for this model. Delete a record to re-arm its cell.

In [ ]:
import runner as R
done = sorted(k for k in R.completed_keys() if k.startswith('static.') and MODEL in k)
print(f'{len(done)} static record(s) for {MODEL}:')
for k in done:
    print(' ', k)